### Organizing and plotting SSC of each grain size class

In [78]:
import pandas as pd
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns  
from scipy import stats

spring_GSD = pd.read_csv('../../data/LISST_data/SP_WC_normalized_GSD_corrected.csv')
summer_GSD = pd.read_csv('../../data/LISST_data/SM_WC_normalized_GSD_corrected.csv')
spring_SSC = pd.read_csv('../../data/constituents/SSC_samples/spring_SSC_samples.csv')
summer_SSC = pd.read_csv('../../data/constituents/SSC_samples/summer_SSC_samples.csv')
tau = pd.read_csv('../../data/shear_stress/average_total_shear_stress.csv')

In [79]:
size_classes_mm = {
    "Clay SSC (mg/L)": (0, 0.002),        # < 0.002 mm
    "Silt SSC (mg/L)": (0.002, 0.063),    # 0.002–0.063 mm
    "Fine sand SSC (mg/L)": (0.063, 0.85) # 0.063–0.85 mm
}

def add_size_class_ssc(ssc_df, gsd_df, size_col="size", labid_col="LabID", ssc_col="SSC (mg/L)"):
    """
    adds clay, silt, and fine sand SSC columns to an SSC dataframe using normalized GSD proportions from LISST data
    """
    ssc_out = ssc_df.copy()
    gsd = gsd_df.copy()
    gsd["size_mm"] = pd.to_numeric(gsd[size_col], errors="coerce") / 1000     # convert LISST size from um to mm


    ssc_out[labid_col] = ssc_out[labid_col].astype(str)
    gsd_labid_cols = [col for col in gsd.columns if col not in [size_col, "size_mm"]]
    # force GSD LabID columns to strings
    gsd = gsd.rename(columns={col: str(col) for col in gsd_labid_cols})
    gsd_labid_cols = [str(col) for col in gsd_labid_cols]

    for class_name, (lower_mm, upper_mm) in size_classes_mm.items():
        # clay is < 0.002 mm; other classes are lower-inclusive, upper-exclusive
        if lower_mm == 0:
            mask = gsd["size_mm"] < upper_mm
        else:
            mask = (gsd["size_mm"] >= lower_mm) & (gsd["size_mm"] < upper_mm)
        # sum GSD proportions in that class for each LabID
        class_fraction = gsd.loc[mask, gsd_labid_cols].sum(axis=0)

        # match each SSC sample to its GSD class fraction
        frac_col = class_name.replace("SSC", "fraction")
        ssc_out[frac_col] = ssc_out[labid_col].map(class_fraction)

        # convert fraction to SSC concentration
        ssc_out[class_name] = ssc_out[ssc_col] * ssc_out[frac_col]

    return ssc_out


# apply to spring and summer
spring_SSC_classes = add_size_class_ssc(spring_SSC, spring_GSD)
summer_SSC_classes = add_size_class_ssc(summer_SSC, summer_GSD)

print(spring_SSC_classes.head())

          DateTime LabID  SSC (mg/L) Reach  Clay fraction (mg/L)  \
0  4/15/2023 12:30   195   12.380952  Down                   NaN   
1  4/15/2023 16:30   196    9.473684  Down                   NaN   
2  4/15/2023 20:30   197    5.000000  Down                   NaN   
3   4/16/2023 0:30   198    4.000000  Down                   NaN   
4  4/16/2023 12:30   199   10.476190  Down                   NaN   

   Clay SSC (mg/L)  Silt fraction (mg/L)  Silt SSC (mg/L)  \
0              NaN                   NaN              NaN   
1              NaN                   NaN              NaN   
2              NaN                   NaN              NaN   
3              NaN                   NaN              NaN   
4              NaN                   NaN              NaN   

   Fine sand fraction (mg/L)  Fine sand SSC (mg/L)  
0                        NaN                   NaN  
1                        NaN                   NaN  
2                        NaN                   NaN  
3           

In [80]:
# export as csv
spring_SSC_classes.to_csv("spring_SSC_with_size_class_concentrations.csv", index=False)
summer_SSC_classes.to_csv("summer_SSC_with_size_class_concentrations.csv", index=False)

### Plotting time series

In [81]:
def clean_event_name(name):
    cleaned = name.replace("_down", "").replace("_up", "")
    cleaned = cleaned.replace("down_", "").replace("up_", "")
    return cleaned

def plot_windows_by_reach(ssc_df, hydro_df, windows, season_label, hydro_time_col="datetime", hydro_y_col="shear_stress", 
                            hydro_y_label="Shear stress (Pa)", buffer_minutes=30, out_dir=None):
    ssc_df = ssc_df.sort_values("DateTime").copy()
    hydro_df = hydro_df.sort_values(hydro_time_col).copy()
    buffer = pd.Timedelta(minutes=buffer_minutes)

    if out_dir is not None:
        os.makedirs(out_dir, exist_ok=True)

    for window_name, (start, end) in windows.items():
        start = pd.Timestamp(start)
        end = pd.Timestamp(end)

        # get reach info from window name
        if "down" in window_name.lower():
            reach_val = "Down"
            reach_title = "Downstream"
        elif "up" in window_name.lower():
            reach_val = "Up"
            reach_title = "Upstream"
        else:
            reach_val = None
            reach_title = "All Reaches"

        # subset SSC by time
        event_ssc = ssc_df[(ssc_df["DateTime"] >= start) & (ssc_df["DateTime"] <= end)].copy()
        # subset by reach
        if reach_val is not None:event_ssc = event_ssc[event_ssc["Reach"].str.strip().str.lower() == reach_val.lower()]
        # subset hydro data
        event_hydro = hydro_df[(hydro_df[hydro_time_col] >= start) & (hydro_df[hydro_time_col] <= end)].copy()
        if event_ssc.empty:
            print(f"Skipping {window_name}: no SSC samples for {reach_title}.")
            continue

        # use sample availability, not full storm duration, for plot limits
        plot_start = event_ssc["DateTime"].min() - buffer
        plot_end = event_ssc["DateTime"].max() + buffer
        # subset shear stress only to the sample-based plotting window
        event_hydro = hydro_df[(hydro_df[hydro_time_col] >= plot_start) & (hydro_df[hydro_time_col] <= plot_end)].copy()

        fig, ax1 = plt.subplots(figsize=(10, 4))
        # left axis is shear stress
        if not event_hydro.empty:
            ax1.plot(event_hydro[hydro_time_col], event_hydro[hydro_y_col], color="cornflowerblue", lw=1.2)
        ax1.set_ylabel(hydro_y_label, color="steelblue")
        ax1.tick_params(axis="y", labelcolor="black")
        ax1.set_xlabel("Date Time")

        # right axis is SSC and size classes
        ax2 = ax1.twinx()

        if not event_ssc.empty:
            ax2.plot(event_ssc["DateTime"], event_ssc["SSC (mg/L)"], color="mediumseagreen", lw=1.2, label="Total SSC (mg/L)")
            ax2.scatter(event_ssc["DateTime"], event_ssc["Clay SSC (mg/L)"], color="red", s=25, marker="o", label="Clay (mg/L)")
            ax2.scatter(event_ssc["DateTime"], event_ssc["Silt SSC (mg/L)"], color="purple", s=25, marker="x", label="Silt (mg/L)")
            ax2.scatter(event_ssc["DateTime"], event_ssc["Fine sand SSC (mg/L)"], color="yellow", s=30, marker="^", label="Fine Sand (mg/L)")


        ax2.set_ylabel("Suspended Sediment Concentration (mg/L)", color="mediumseagreen")
        ax2.tick_params(axis="y", labelcolor="black")
        base_event = clean_event_name(window_name)
        ax1.set_title(f"{season_label} - {base_event} - {reach_title}", fontsize=10)
        ax2.legend(loc="upper left", fontsize=9, frameon=True)
        plt.tight_layout()

        if out_dir is not None:
            fname = f"{season_label}_{window_name}".replace(" ", "_").replace("/", "_")
            plt.savefig(os.path.join(out_dir, f"{fname}.png"), dpi=300, bbox_inches="tight")
        plt.close(fig)

In [82]:
# make sure datetime columns are datetime
spring_SSC_classes["DateTime"] = pd.to_datetime(spring_SSC_classes["DateTime"])
summer_SSC_classes["DateTime"] = pd.to_datetime(summer_SSC_classes["DateTime"])
tau["datetime"] = pd.to_datetime(tau["datetime"])

Summer Storms

In [83]:
storm_windows = {
    "st1_down": ("2021-07-23 12:30", "2021-07-28 11:15"),
    "st1_up":   ("2021-07-23 12:30", "2021-07-28 11:15"),
    "st2_down": ("2022-08-03 14:30", "2022-08-03 18:30"),
    "st2_up":   ("2022-08-03 14:30", "2022-08-03 18:30"),
    "st3_down": ("2022-08-08 12:30", "2022-08-08 22:15"),
    "st3_up":   ("2022-08-08 12:30", "2022-08-08 22:15"),
    "st4_down": ("2023-07-29 13:00", "2023-07-30 10:30"),
    "st4_up":   ("2023-07-29 13:00", "2023-07-30 10:30"),
    "st5_down": ("2023-08-13 17:15", "2023-08-14 03:15"),
    "st5_up":   ("2023-08-13 17:15", "2023-08-14 03:15"),
    "st6_down": ("2023-08-28 11:30", "2023-08-28 16:30"),
    "st6_up":   ("2023-08-28 11:30", "2023-08-28 16:30"),
    "st7_down": ("2023-09-14 13:00", "2023-09-15 13:00"),
    "st7_up":   ("2023-09-14 13:00", "2023-09-15 13:00")
}

# plot
plot_windows_by_reach(ssc_df=summer_SSC_classes, hydro_df=tau, windows=storm_windows, season_label="Summer", hydro_time_col="datetime", hydro_y_col="shear_stress", 
                    hydro_y_label="Shear stress", out_dir="plots/time_series")


Skipping st2_up: no SSC samples for Upstream.
Skipping st3_up: no SSC samples for Upstream.
Skipping st6_down: no SSC samples for Downstream.


Spring Events

In [84]:
event_windows = {
    "down_event1": ("2023-04-17 13:00", "2023-04-18 11:00"),
    "down_event2": ("2023-04-18 11:00", "2023-04-19 10:00"),
    "down_event3": ("2023-04-19 10:00", "2023-04-20 11:00"),
    "down_event4": ("2023-04-21 12:00", "2023-04-22 09:30"),
    "down_event5": ("2023-04-22 14:00", "2023-04-23 10:30"),
    "down_event6": ("2023-04-23 11:30", "2023-04-24 11:00"),
    "down_event7": ("2023-04-24 11:00", "2023-04-25 11:00"),
    "down_event8": ("2023-04-27 12:00", "2023-04-28 12:00"),
    "down_event9": ("2023-04-29 12:00", "2023-04-30 12:00"),
    "down_event10": ("2023-04-30 12:00", "2023-05-01 11:00"),
    "down_event11": ("2023-05-01 11:00", "2023-05-02 13:00"),
    "down_event12": ("2023-05-02 13:00", "2023-05-03 11:30"),
    "down_event13": ("2023-05-03 11:30", "2023-05-04 10:00"),
    "down_event14": ("2023-05-04 11:00", "2023-05-05 09:00"),
    "down_event15": ("2023-05-05 10:00", "2023-05-06 09:00"),
    "up_event1": ("2023-04-17 13:00", "2023-04-18 11:00"),
    "up_event2": ("2023-04-18 11:00", "2023-04-19 10:00"),
    "up_event3": ("2023-04-19 10:00", "2023-04-20 11:00"),
    "up_event4": ("2023-04-21 12:00", "2023-04-22 09:30"),
    "up_event5": ("2023-04-22 14:00", "2023-04-23 10:30"),
    "up_event6": ("2023-04-23 11:30", "2023-04-24 11:00"),
    "up_event7": ("2023-04-24 11:00", "2023-04-25 11:00"),
    "up_event8": ("2023-04-27 12:00", "2023-04-28 12:00"),
    "up_event9": ("2023-04-29 12:00", "2023-04-30 12:00"),
    "up_event10": ("2023-04-30 12:00", "2023-05-01 11:00"),
    "up_event11": ("2023-05-01 11:00", "2023-05-02 13:00"),
    "up_event12": ("2023-05-02 13:00", "2023-05-03 11:30"),
    "up_event13": ("2023-05-03 11:30", "2023-05-04 10:00"),
    "up_event14": ("2023-05-04 11:00", "2023-05-05 09:00"),
    "up_event15": ("2023-05-05 10:00", "2023-05-06 09:00"),
}

# spring events
plot_windows_by_reach(ssc_df=spring_SSC_classes, hydro_df=tau, windows=event_windows, season_label="Spring", hydro_time_col="datetime", hydro_y_col="shear_stress",
                    hydro_y_label="Shear stress", out_dir="plots/time_series")

Skipping down_event4: no SSC samples for Downstream.
Skipping down_event5: no SSC samples for Downstream.
Skipping down_event15: no SSC samples for Downstream.
